### Install new libraries

In [32]:
#!pip install ddgs trafilatura -q # -q without any logs
#!pip install openai-agents
#!pip install boto3

### Step 0: Setup imports

In [33]:
import os
from dotenv import load_dotenv
from pprint import pprint
import json
from ddgs import DDGS
import trafilatura
from IPython.display import Image, display, Markdown
from openai import OpenAI
from agents import trace, Agent, Runner, function_tool, handoff
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
import base64
import boto3
import uuid
import io

load_dotenv()
openai_client = OpenAI()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API Key is Missing")

MODEL="gpt-4o-mini"
#MODEL="gpt-5.6-luna"

### Step 1: Function tools

In [34]:
@function_tool
def search_web(query: str):
    """ Search the web using DuckDuckGo browser. Return 3 results."""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=10)
    print(f"\u2705 search_web: Got results {query}")
    return json.dumps(results, indent=2)

In [35]:
@function_tool
def fetch_url(url: str):
    """Fetch the content of a URL using trafilatura"""
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f"\u2705 fetch_url: Got {len(text)} chars from {url[:60]}")
            return text
    print(f"\u274c Failed to fetch or extract test fron {url}.")
    return f"Could not extract the text from {url}. try a different source"

In [36]:
if not os.getenv("AWS_S3_BUCKET"):
    raise RuntimeError("AWS_S3_BUCKET is missing")

s3_client = boto3.client("s3")
s3_bucket = os.environ["AWS_S3_BUCKET"]


@function_tool
def generate_image(prompt: str) -> str:
    """"Generate an image using OpenAI's gpt-image-2 API, upload it to S3, and return its private URL. The prompt should be a detailed visual description"""

    print(f"🎨 Generating image for: {prompt}")

    response = openai_client.images.generate(
        model="gpt-image-2",
        prompt=prompt,
        size="1024x1024",
        quality="high",
        n=1,
    )

    image_base64 = response.data[0].b64_json

    if not image_base64:
        raise RuntimeError("The image API did not return image data")

    image_bytes = base64.b64decode(image_base64)
    object_key = f"generated-images/{uuid.uuid4()}.png"

    # Upload directly from memory — no local file is created.
    s3_client.upload_fileobj(
        io.BytesIO(image_bytes),
        s3_bucket,
        object_key,
        ExtraArgs={
            "ContentType": "image/png",
            "ContentDisposition": "inline",
        },
    )

    # Private URL valid for one hour.
    image_url = s3_client.generate_presigned_url(
        "get_object",
        Params={
            "Bucket": s3_bucket,
            "Key": object_key,
        },
        ExpiresIn=3600,
    )

    print(f"✅ Image uploaded: {image_url}")
    return image_url


### Step 2: The Agent Prompt tells the LLM *who it is* and *how to behave*. 
#### The key things:
- what its job is
- What tool it has
- what process to follow
- what output format to produce

#### Research Agent

In [37]:
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

***IMPORTANT:
After each search with search_web, you MUST first explain your reasoning:
- Which URLs look most relevant and why
- Which ones you will fetch and why
- Which ones you are skipping and why
Only AFTER writing out your reasoning should you call fetch_url.***

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize into a research brief

You MUST gather information from at least 3 distinct sources before delivering your brief. 
If you have fewer than 3 sources, keep searching.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""


research_agent = Agent(
    name = "Research Agent",
    instructions = RESEARCH_AGENT_PROMPT,
    model = MODEL,
    tools = [search_web, fetch_url]
)

### Image Generatrion Agent

In [38]:
IMAGE_AGENT_PROMPT = """You are an image prompt specialist. Given a topic and content summary,
craft a detailed gpt-image-2 prompt for a hero image.

Rules for your gpt-image-2 prompt:
- Describe a natural, photographic-style image (not illustrated, not cartoon)
- No text, logos, or words in the image
- No human faces or recognizable people
- No icon dumps or collages
- Focus on a single compelling visual that captures the theme
- Be specific about lighting, composition, and mood
- Keep the prompt under 200 words

Call generate_image exactly ONCE with your prompt. One image only.
"""

image_agent = Agent(
    name = "Image Agent",
    instructions = IMAGE_AGENT_PROMPT,
    model = MODEL,
    tools = [generate_image]
)

image_agent_as_tool = image_agent.as_tool(tool_name="image_agent", tool_description="Generate a hero image for the article, pass the topic and content summary as input")

### Orchestrator Agent

In [39]:
ORCHESTRATOR_AGENT_PROMPT = RECOMMENDED_PROMPT_PREFIX + """
You are the orchestrator of a multi-agent article writing system.
Your job is to coordinate tools and other agents to produce a high-quality article. 
Use the tools available to you and/or delegate tasks to the appropriate agents.
Never do the work yourself. Always use tools or agents. 
Your tools and agents are specialists and should be doing the work, you are the manager.

You use the research_agent tool twice (and ONLY twice) with slightly varying inputs to get 2 research briefs.
You pick the best research brief out of the two and deliver it as output. 
Do not combine the two briefs, just pick the best one.
Do not do the research yourself or add anything, you MUST use the research_agent tool to get the briefs.

Once you have selected the best brief, you MUST use the image_agent tool to generate an image.
use the research brief to supply the image agent with the topic and content summary it needs to generate the image.

Handoff the research brief to the journalist writer Agent, 
and include the image URL as part of your handoff.

IMPORTANT: When passing the image URL, copy it Exactly as returned by the image_agent tool. Do not modify it in any way.
"""

#Deliver the selected research brief as the final output, and 
#include the image at the top of your final output in Markdown format like this ![hero image](image_url)

orchestrator_agent = Agent(
    name = "orchestrator_agent",
    instructions=ORCHESTRATOR_AGENT_PROMPT,
    #model= "gpt-5-mini",
    model = MODEL,
    tools = [research_agent.as_tool(tool_name="research_agent", tool_description="Research a topic and return a brief with key fscts, statistics, themes, and source URLs, pass the topic as input"),
             image_agent_as_tool]
)

### Writer Agent A: The Journalist

In [40]:
JOURNALIST_WRITER_PROMPT = RECOMMENDED_PROMPT_PREFIX + """You are an investigative journalist. 
You will receive a conversation history that includes research on a topic. 

Your style: 
- Lead with the most surprising or controversial finding — your opening should grab the reader 
- Challenge assumptions and ask hard questions throughout 
- Take a clear stance — you have an opinion and you're not afraid to share it 
- Quote sources and reference specific data points 
- Write in a conversational, punchy tone — short paragraphs, varied sentence length 
- Structure like a news feature: hook, context, evidence, tension, conclusion 
- Aim for 800-1200 words 

Do NOT use generic section headers like "Introduction", "Conclusion", or "Overview". Use creative, specific headers to grab the attention.
Do NOT overuse headers. Only use one title and between 2 and 3 headers in total for the entire article.
Do NOT use Bullet points or numbered lists.
Do NOT ask for feedback, offer revisions, or include any commentary after the article. 
Just deliver the finished article in markdown format. 

The conversation history contains an image_agent result.

Extract the exact Markdown image reference returned by image_agent and
place it immediately after the article title. Preserve the URL exactly,
including its complete query string.

If no image_agent result is present, do not invent an image URL.
"""

journalist_agent = Agent(
    name = "journalist_agent",
    instructions = JOURNALIST_WRITER_PROMPT,
    model = MODEL
)

### Update the Orchestrator Agent

In [41]:
orchestrator_agent.handoffs = [journalist_agent]

### Run -- Agent

In [42]:
with trace("Article Writer with handoff", group_id="Learning AI agents",
           metadata={"topic": "Testing the filters in the traces"}):
    result = await Runner.run(
        orchestrator_agent,
        input = "How one can I transition from Software engineer to AI Engineer?",
        max_turns=10
    )

✅ search_web: Got results career transition from Software Engineering to AI Engineering necessary skills education requirements job market insights resources
✅ search_web: Got results transitioning from Software Engineer to AI Engineer skills education responsibilities trends best practices


CancelledError: 

In [ ]:
print(f"Agent: {result.last_agent.name}")
print(f"-----")
display(Markdown(result.final_output))